# 05 · Casos, conclusiones y valor

**Clases 1, 4 y 12:** transformar información en decisiones, evaluar el objetivo del negocio y entender los residuos.

La aplicación ayuda a priorizar una revisión técnica. La diferencia entre energía esperada y observada es una estimación relativa al modelo: no demuestra cuánta energía se podría recuperar interviniendo.

El precio USD 42,84/MWh es una ilustración de contexto argentino (RenovAr Ronda 2), no el ingreso de las plantas chinas. No se comunica ahorro por reparación sin un experimento que lo sustente.

In [ ]:
from pathlib import Path
import os, sys
if 'google.colab' in sys.modules:
    import subprocess
    root = Path('/content/tpo-solar')
    if not root.exists():
        subprocess.run(['git','clone','https://github.com/imelinc/tpo-solar.git',str(root)],check=True)
    os.chdir(root)
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.config import DATA_RAW, DATA_PROCESSED
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


In [ ]:
import json
D=json.loads((DATA_PROCESSED/'defensa_evidencia.json').read_text())
diario=pd.DataFrame(D['diario'])
s9=diario[diario.plant_id.eq('station09')]
validos=s9[s9.decide]
print('Días con dato:',len(s9),'| días con decisión:',len(validos),'| alertas:',s9.alerta.sum())
print('Energía observada:',(validos.real*validos.capacidad).sum(),'MWh')
print('Referencia del modelo:',(validos.esperada*validos.capacidad).sum(),'MWh')
deficit=validos.loc[validos.alerta,'deficit_mwh'].sum()
print('Déficit estimado en días con alerta:',deficit,'MWh')
print('Valor ilustrativo:',deficit*D['economia']['precio'],'USD')
px.line(s9,x='fecha',y=['real','esperada'],title='station09 · Energía normalizada observada y referencia',
        labels={'value':'MWh por MW nominal'}).show()


In [ ]:
arg=pd.DataFrame(D['argentina'])
u=arg[arg.central.eq('ULL4FV')]
print(u[u.fecha.str.startswith('2025')][['fecha','energia_gwh','indice','decide','alerta']].to_string(index=False))
px.line(u,x='fecha',y=['factor_capacidad','pred'],
        title='ULL4FV · Comparación mensual, sin diagnóstico confirmado').show()
print('Ceros conservados en Argentina:',D['regresion_argentina']['ceros_conservados'])


## Cierre de la hipótesis

La hipótesis de negocio es que comparar la producción observada con una referencia condicionada por clima puede señalar plantas para revisar. El soporte debe leerse en dos partes: calidad de la regresión y desempeño de la regla sobre caídas controladas. Los resultados completos y sus matrices están en 03 y 04 y en el dashboard.

Una sola vuelta estacional, sensores, capacidad declarada y falta de etiquetas operativas limitan la conclusión. Para usarlo en producción hacen falta más años y registros técnicos independientes. Argentina es comparación mensual y no una alarma diaria.

## Exposición compartida (4 integrantes)

Ignacio Melinc, Matias Young, Francisco Fabrello y Malena Garcia Moraglio trabajan en todas las etapas. El guion propone un reparto oral para que participen los cuatro, sin atribuir roles exclusivos de desarrollo.
